# PoultrySense - Detection RETRAIN on D3 + D4_Crowded

Fine-tunes a **YOLOv8n detection** model on the merged dataset:

| Split | D3_Detection (your behavior videos) | D4_Crowded (external repo) |
|---|---|---|
| train | 156 images | 16 images |
| val | 36 images | 4 images |

D4_Crowded comes from
[Chicken-Behavior-Analysis](https://github.com/TaherehZarratEhsan/Chicken-Behavior-Analysis)
(Apache 2.0): 20 dense, wide-angle 1920x1080 broiler-pen frames, converted to YOLO
format by `datasets/build_d4_crowded.py`. Its purpose: teach the detector to find
chickens in the **wide-angle crowded pen view**, where the close-up-trained
segmentation model under-detects (tracking recall ~0.03 on job_000008).

> Keeps YOLOv8 (no YOLOv4 / no YOLOv11). job_000011 stays the held-out test video.
>
> **Estimated runtime on T4:** ~25-45 min (60 epochs, imgsz=960, batch=16).


In [ ]:
# 1) Install (idempotent) - same as Part A of PoultrySense_Colab.ipynb
%pip install -q ultralytics
from ultralytics import YOLO
import torch, os, zipfile, shutil, glob, cv2
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## Step 1 - upload the two zips to Drive

Before running the next cell, upload to your Drive folder `MyDrive/PoultrySense/`:

1. `D3_Detection.zip`   (already in `datasets/`)
2. `D4_Crowded.zip`     (already in `datasets/`, built by `build_d4_crowded.py`)

Then run the cell below and grant Colab access to Drive.
If your folder is not named `PoultrySense`, edit `DRIVE`.

In [ ]:
# 2) Mount Drive and point to your PoultrySense folder
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/PoultrySense'
ZIPS  = ['D3_Detection.zip', 'D4_Crowded.zip']
DATA  = '/content/datasets'
COMBINED = f'{DATA}/D3D4_Detection/data.yaml'

for z in ZIPS:
    p = f'{DRIVE}/{z}'
    assert os.path.exists(p), f'Missing {p} - upload it to Drive first'
    print('found', z, f'{os.path.getsize(p)/1e6:.1f} MB')

## Step 2 - extract both datasets

Unzips both zips under `/content/datasets` and writes the merged `data.yaml`
(the one committed at `datasets/D3D4_Detection/data.yaml`).

In [ ]:
# 3) Extract D3 + D4 and write the merged data.yaml
os.makedirs(DATA, exist_ok=True)
for z in ZIPS:
    with zipfile.ZipFile(f'{DRIVE}/{z}') as zf:
        zf.extractall(DATA)

yml = '''train:
  - /content/datasets/D3_Detection/train/images
  - /content/datasets/D4_Crowded/train/images
val:
  - /content/datasets/D3_Detection/val/images
  - /content/datasets/D4_Crowded/val/images
nc: 1
names: ['chicken']
'''
os.makedirs(os.path.dirname(COMBINED), exist_ok=True)
with open(COMBINED, 'w') as f:
    f.write(yml)

nt = len(glob.glob(f'{DATA}/D3_Detection/train/images/*.jpg')) + \
     len(glob.glob(f'{DATA}/D4_Crowded/train/images/*.png'))
nv = len(glob.glob(f'{DATA}/D3_Detection/val/images/*.jpg')) + \
     len(glob.glob(f'{DATA}/D4_Crowded/val/images/*.png'))
print(f'merged train images: {nt}, val images: {nv}')

## Step 3 - sanity-check the data

Renders a few labeled crops so you can eyeball the boxes before training.

In [ ]:
# 4) Show sample labeled images from each dataset
def show_sample(image_dir, label_dir, fname, title):
    img = cv2.imread(os.path.join(image_dir, fname))
    H, W = img.shape[:2]
    stem = os.path.splitext(fname)[0]
    lab = os.path.join(label_dir, f'{stem}.txt')
    if os.path.exists(lab):
        for line in open(lab):
            _, cx, cy, w, h = map(float, line.split())
            x1 = int((cx - w/2) * W); y1 = int((cy - h/2) * H)
            x2 = int((cx + w/2) * W); y2 = int((cy + h/2) * H)
            cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 1)
    from google.colab.patches import cv2_imshow
    print(title)
    cv2_imshow(img)

show_sample(f'{DATA}/D3_Detection/train/images', f'{DATA}/D3_Detection/train/labels',
            sorted(os.listdir(f'{DATA}/D3_Detection/train/images'))[0], 'D3 sample')
show_sample(f'{DATA}/D4_Crowded/train/images', f'{DATA}/D4_Crowded/train/labels',
            sorted(os.listdir(f'{DATA}/D4_Crowded/train/images'))[0], 'D4_Crowded sample')

## Step 4 - fine-tune yolov8n detection

Same recipe as `colab_retrain_d3.py` (SGD lr0=0.01, imgsz=960, patience=50) but
fine-tuned (60 epochs, warmup) on the **merged** D3+D4 set so the crowd frames
help without over-fitting. Adjust `EPOCHS` if you want a longer run.

In [ ]:
# 5) Train
EPOCHS = 60
model = YOLO('yolov8n.pt')
r = model.train(data=COMBINED,
                epochs=EPOCHS, imgsz=960, batch=16, device=0,
                optimizer='SGD', lr0=0.01, cache=True,
                project='/content/runs', name='det_d3d4',
                patience=50, seed=0)
print('Final train metrics (D3+D4 detection):')
print(r.results_dict)

## Step 5 - save the model to Drive

Copies the best checkpoint to `MyDrive/PoultrySense/models/` as
`PoultrySense_YOLOv8_ChickenDet.pt` (do not overwrite the D1 seg model).

In [ ]:
# 6) Copy best.pt to Drive as the new detection model
best = '/content/runs/det_d3d4/weights/best.pt'
dst = f'{DRIVE}/models/PoultrySense_YOLOv8_ChickenDet.pt'
os.makedirs(os.path.dirname(dst), exist_ok=True)
shutil.copy(best, dst)
print('Saved to', dst, f'{os.path.getsize(dst)/1e6:.1f} MB')

## Step 6 - validation report

Runs the saved model against the merged val set (includes the 4 D4 crowd frames).

In [ ]:
# 7) Validation report on merged val set
m = YOLO(best).val(data=COMBINED)
print('val (box): mAP50=%.4f  mAP50-95=%.4f  P=%.4f  R=%.4f'
      % (m.box.map50, m.box.map, m.box.mp, m.box.mr))

## Next steps (back in the repo)

1. Download `PoultrySense_YOLOv8_ChickenDet.pt` into `models/`.
2. Run `python pipeline/test_tracking_job_000008.py` with the new model and compare
   recall / mean IoU / ID switches against the baseline
   `output/tracking_000008_results.json` (recall 0.0321, IoU 0.5314).
3. If improved, point `pipeline.py` `SEG_MODEL` at the new detection model
   (tracking only needs boxes) and regenerate the annotated video + JSON.